# SROIE OCR Preprocessing Experiments

This notebook runs the existing Tesseract OCR engine over the same validated SROIE samples for three lightweight Pillow preprocessing variants. The baseline JSONL is read but never overwritten, raw images are opened read-only, and no field extraction or model training is performed.

In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.sroie import load_sroie_dataset
from src.ocr import (
    TesseractOCREngine,
    enhance_contrast,
    grayscale_image,
    resize_image,
    run_batch_ocr,
    threshold_image,
)

DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
BASELINE_PATH = PROCESSED_ROOT / 'sroie_ocr.jsonl'
FIELDS = ['company', 'date', 'address', 'total']

dataset = load_sroie_dataset(DATASET_ROOT, strict=False)
records = dataset.to_dict('records')
assert len(records) == 624, f'Expected 624 validated samples, found {len(records)}'
if not BASELINE_PATH.is_file():
    raise FileNotFoundError(f'Baseline OCR results not found: {BASELINE_PATH}')

print(f'Validated samples used in every configuration: {len(records)}')
print(f'Baseline preserved at: {BASELINE_PATH.relative_to(PROJECT_ROOT)}')

## Preprocessing configurations

Each callable returns a new Pillow image. The OCR engine and batch writer remain the shared project utilities, so image IDs, statuses, words, and text follow the same JSONL schema as the baseline.

In [ ]:
def gray_contrast(image: Image.Image) -> Image.Image:
    return enhance_contrast(grayscale_image(image), factor=2.0)

def combined_preprocessing(image: Image.Image) -> Image.Image:
    grayscale = grayscale_image(image)
    resized = resize_image(grayscale, scale=2.0)
    return threshold_image(resized, threshold=160)

CONFIGURATIONS = {
    'resize': {
        'preprocess': lambda image: resize_image(image, scale=2.0),
        'output_path': PROCESSED_ROOT / 'sroie_ocr_preprocessed_resize.jsonl',
    },
    'grayscale_contrast': {
        'preprocess': gray_contrast,
        'output_path': PROCESSED_ROOT / 'sroie_ocr_preprocessed_gray_contrast.jsonl',
    },
    'combined': {
        'preprocess': combined_preprocessing,
        'output_path': PROCESSED_ROOT / 'sroie_ocr_preprocessed_combined.jsonl',
    },
}

for name, configuration in CONFIGURATIONS.items():
    preview = configuration['preprocess'](Image.new('RGB', (20, 10), 'white'))
    print(f"{name}: mode={preview.mode}, size={preview.size}, output={configuration['output_path'].name}")

## Run OCR

This writes one separate ignored JSONL file per variant. Re-running this cell refreshes only the experiment outputs and never writes to `sroie_ocr.jsonl`.

In [ ]:
experiment_summaries = {}
for name, configuration in CONFIGURATIONS.items():
    engine = TesseractOCREngine(preprocess=configuration['preprocess'])
    summary = run_batch_ocr(records, configuration['output_path'], engine=engine)
    experiment_summaries[name] = summary
    print(
        f"{name}: success={summary.success}, empty={summary.empty}, "
        f"failure={summary.failure}, output={summary.output_path.relative_to(PROJECT_ROOT)}"
    )

## Coverage evaluation

The comparison uses the same evaluation-only normalization as the previous error-analysis notebook: case folding, whitespace/line-break removal, and decimal comma-to-point normalization. It checks whether each normalized ground-truth value occurs in the OCR text; it does not extract fields.

In [ ]:
def load_ocr_jsonl(path: Path) -> pd.DataFrame:
    with path.open(encoding='utf-8') as stream:
        records_from_file = [json.loads(line) for line in stream if line.strip()]
    return pd.DataFrame.from_records(records_from_file)

def normalize_for_match(value: object) -> str:
    text = str(value) if value is not None else ''
    text = text.casefold().replace(',', '.')
    return re.sub(r'\s+', '', text)

def evaluate_ocr(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    ocr = load_ocr_jsonl(path)
    required = {'image_id', 'status', 'text'}
    missing = required - set(ocr.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    ocr_by_id = ocr.drop_duplicates('image_id', keep='first').set_index('image_id')
    joined = dataset[['image_id', *FIELDS]].copy()
    joined['ocr_text'] = joined['image_id'].map(ocr_by_id['text']).fillna('')
    joined['ocr_status'] = joined['image_id'].map(ocr_by_id['status']).fillna('missing')
    coverage_rows = []
    for field in FIELDS:
        valid = joined[field].fillna('').astype(str).str.strip().ne('')
        matches = [
            bool(normalize_for_match(truth)) and normalize_for_match(truth) in normalize_for_match(ocr_text)
            for truth, ocr_text in zip(joined[field], joined['ocr_text'])
        ]
        valid_count = int(valid.sum())
        matched_count = int(sum(match and is_valid for match, is_valid in zip(matches, valid)))
        coverage_rows.append({
            'field': field,
            'coverage_percent': round(100 * matched_count / valid_count, 2) if valid_count else 0.0,
            'valid_samples': valid_count,
            'matched_samples': matched_count,
        })
    coverage = pd.DataFrame(coverage_rows).set_index('field')
    return joined, coverage

evaluation = {}
baseline_records, baseline_coverage = evaluate_ocr(BASELINE_PATH)
evaluation['baseline'] = {'records': baseline_records, 'coverage': baseline_coverage}
for name, configuration in CONFIGURATIONS.items():
    records_for_config, coverage_for_config = evaluate_ocr(configuration['output_path'])
    evaluation[name] = {'records': records_for_config, 'coverage': coverage_for_config}

comparison = pd.DataFrame({
    name: evaluation[name]['coverage']['coverage_percent']
    for name in ['baseline', *CONFIGURATIONS]
}).T
comparison.index.name = 'configuration'
display(comparison[FIELDS])

## OCR status comparison

Status counts are read from each actual JSONL output rather than inferred from coverage.

In [ ]:
status_rows = []
for name, result in evaluation.items():
    counts = result['records']['ocr_status'].value_counts()
    status_rows.append({
        'configuration': name,
        'success': int(counts.get('success', 0)),
        'empty': int(counts.get('empty', 0)),
        'failure': int(counts.get('failure', 0)),
    })
status_comparison = pd.DataFrame(status_rows).set_index('configuration')
display(status_comparison)

## Best configuration by field

The summary below reports the highest measured coverage independently for each field. It intentionally does not select an overall winner.

In [ ]:
for field in FIELDS:
    best_value = comparison[field].max()
    best_configurations = comparison.index[comparison[field].eq(best_value)].tolist()
    print(f"{field}: {', '.join(best_configurations)} ({best_value:.2f}%)")